<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/headlineGenerationNLP_BART_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BART Training Notebook — News Headline Generation

This notebook trains **facebook/bart-base** for headline generation using the model-ready files created by the preprocessing notebook.

Expected files from preprocessing:

- `bart_base_train.csv`
- `bart_base_validation.csv`
- `bart_base_test.csv`

Each file should contain:

- `model_input`: cleaned article text with `summarize:` prefix
- `model_target`: cleaned headline/title

Recommended Colab runtime from your screenshot: choose **A100 GPU** or **H100 GPU** if available. If not, use **T4 GPU** and reduce batch size / sample size.

## 1. Install Required Packages

Run this once at the start of the Colab session.

The `sympy` line helps avoid the common `AttributeError: module 'sympy' has no attribute 'core'` error that can happen with some PyTorch/Transformers imports.

In [5]:
!pip -q install -U transformers datasets accelerate evaluate rouge_score sentencepiece sacrebleu
!pip -q install -U sympy

print('Packages installed.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.7 MB/s eta 0:00:00
Packages installed.


## 2. Imports and GPU Check

In [6]:
import os
import gc
import json
import math
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
    set_seed,
)
import evaluate

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU memory GB: 39.49


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Mount Google Drive and Set Paths

This uses the same project folder as the preprocessing notebook.

In [7]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running in Colab or Drive mount skipped.')

BASE_DIR = '/content/drive/MyDrive/headlineGenerationProjectNLP'
DATA_DIR = f'{BASE_DIR}/news_headline_model_data'
OUTPUT_DIR = f'{BASE_DIR}/bart_headline_model'

if not os.path.exists('/content/drive/MyDrive'):
    DATA_DIR = '/mnt/data/news_headline_model_data'
    OUTPUT_DIR = '/mnt/data/bart_headline_model'

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = f'{DATA_DIR}/bart_base_train.csv'
VAL_PATH   = f'{DATA_DIR}/bart_base_validation.csv'
TEST_PATH  = f'{DATA_DIR}/bart_base_test.csv'

print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Train exists:', os.path.exists(TRAIN_PATH), TRAIN_PATH)
print('Val exists:', os.path.exists(VAL_PATH), VAL_PATH)
print('Test exists:', os.path.exists(TEST_PATH), TEST_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data
OUTPUT_DIR: /content/drive/MyDrive/headlineGenerationProjectNLP/bart_headline_model
Train exists: True /content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/bart_base_train.csv
Val exists: True /content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/bart_base_validation.csv
Test exists: True /content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data/bart_base_test.csv


## 4. Load BART Model-Ready Data

The preprocessing notebook saves BART-specific files with standardized column names: `model_input` and `model_target`.

In [8]:
def load_split(path, split_name):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{split_name} file not found: {path}\n'
            'Run the preprocessing notebook first and make sure it saved bart_base_train/validation/test.csv.'
        )
    df = pd.read_csv(path)
    required = {'model_input', 'model_target'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{split_name} is missing columns: {missing}. Found: {df.columns.tolist()}')
    df = df.dropna(subset=['model_input', 'model_target']).copy()
    df['model_input'] = df['model_input'].astype(str).str.strip()
    df['model_target'] = df['model_target'].astype(str).str.strip()
    df = df[(df['model_input'] != '') & (df['model_target'] != '')].copy()
    print(split_name, df.shape)
    return df

train_df = load_split(TRAIN_PATH, 'train')
val_df = load_split(VAL_PATH, 'validation')
test_df = load_split(TEST_PATH, 'test')

display(train_df.head(3))

train (38076, 2)
validation (4760, 2)
test (4760, 2)


,model_input,model_target
0,summarize: Samsung's Galaxy S8 blends beautifu...,It sure looks like the Galaxy S8's battery won...
1,summarize: The almost unbearably excellent sho...,Five Centuries of Drawings at the Morgan
2,"summarize: This week, Rep. Joaquin CastroJoaqu...",Retaliation tweets like Castro's to discourage...


## 5. Optional: Use a Smaller Training Subset

Use this if you only have around **6 hours** or if Colab disconnects.

- For **H100/A100**: you can keep `USE_SUBSET = False`.
- For **T4/G4**: start with `USE_SUBSET = True`, then increase the number later.

In [9]:
USE_SUBSET = False
TRAIN_SAMPLES = 20000
VAL_SAMPLES = 2000
TEST_SAMPLES = 2000

if USE_SUBSET:
    train_df = train_df.sample(min(TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(min(VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(min(TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print('Train:', train_df.shape)
print('Validation:', val_df.shape)
print('Test:', test_df.shape)

Train: (38076, 2)
Validation: (4760, 2)
Test: (4760, 2)


## 6. Convert DataFrames to Hugging Face Datasets

In [10]:
train_ds = Dataset.from_pandas(train_df[['model_input', 'model_target']], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['model_input', 'model_target']], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['model_input', 'model_target']], preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['model_input', 'model_target'],
    num_rows: 38076
})
Dataset({
    features: ['model_input', 'model_target'],
    num_rows: 4760
})
Dataset({
    features: ['model_input', 'model_target'],
    num_rows: 4760
})


## 7. Load BART Tokenizer and Model

In [11]:
MODEL_NAME = 'facebook/bart-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Saves memory during training, useful for T4 and long inputs.
model.gradient_checkpointing_enable()
model.config.use_cache = False

print('Loaded:', MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loaded: facebook/bart-base


## 8. Tokenization

BART can handle up to 1024 input tokens. For headline generation, we keep the article up to 1024 tokens and the headline up to 64 tokens.

In [12]:
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 64

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch['model_input'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=batch['model_target'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
tokenized_test = test_ds.map(preprocess_function, batched=True, remove_columns=test_ds.column_names)

print(tokenized_train)

Map:   0%|          | 0/38076 [00:00<?, ? examples/s]

Map:   0%|          | 0/4760 [00:00<?, ? examples/s]

Map:   0%|          | 0/4760 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 38076
})


## 9. Data Collator

This pads each batch dynamically instead of padding all examples to 1024 tokens, which saves GPU memory.

In [13]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

## 10. ROUGE Metric

In [18]:
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    predictions = np.where(
        (predictions < 0) | (predictions >= tokenizer.vocab_size),
        tokenizer.pad_token_id,
        predictions
    )

    labels = np.where(
        labels == -100,
        tokenizer.pad_token_id,
        labels
    )

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }

## 11. Training Settings

Use these safe settings first.

For T4, keep batch size small. For A100/H100, you can increase `per_device_train_batch_size` to 4 or 8.

In [19]:
fp16_enabled = torch.cuda.is_available()
bf16_enabled = False

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0).lower()

    # A100/H100 support bf16 well
    bf16_enabled = ('a100' in gpu_name) or ('h100' in gpu_name)

    # Use fp16 only if bf16 is disabled
    fp16_enabled = not bf16_enabled

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=2,
    learning_rate=3e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,

    eval_strategy='steps',
    save_strategy='steps',

    eval_steps=1000,
    save_steps=1000,
    logging_steps=100,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,

    fp16=fp16_enabled,
    bf16=bf16_enabled,

    report_to='none',
    remove_unused_columns=True,
)

print('fp16:', fp16_enabled, 'bf16:', bf16_enabled)
print(training_args)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


fp16: False bf16: True
Seq2SeqTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1000,
eval_strategy

## 12. Create Trainer

In [20]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

## 13. Train BART

If training stops because Colab disconnects, rerun the notebook and set `resume_from_checkpoint=True` if a checkpoint exists.

In [21]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Drive folder where the final trained model will be saved
DRIVE_MODEL_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/bart_headline_generation_model/final_model"

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

RESUME_FROM_CHECKPOINT = False

train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

# Save final trained model directly to Google Drive
trainer.save_model(DRIVE_MODEL_DIR)
tokenizer.save_pretrained(DRIVE_MODEL_DIR)

print("Training finished.")
print("Model and tokenizer saved directly to Drive here:")
print(DRIVE_MODEL_DIR)
print(train_result)

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1000,12.838259,1.995823,0.427863,0.206864,0.390003
2000,17.274102,1.875655,0.436713,0.215967,0.398627
3000,14.785872,1.877233,0.439477,0.218241,0.402005
4000,14.475435,1.850299,0.445156,0.222382,0.405574
4760,14.413197,1.839970,0.446843,0.223274,0.407352


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished.
Model and tokenizer saved directly to Drive here:
/content/drive/MyDrive/headlineGenerationProjectNLP/bart_headline_generation_model/final_model
TrainOutput(global_step=4760, training_loss=14.872376578194755, metrics={'train_runtime': 6846.8338, 'train_samples_per_second': 11.122, 'train_steps_per_second': 0.695, 'total_flos': 3.108875664900096e+16, 'train_loss': 14.872376578194755, 'epoch': 2.0})


## 14. Evaluate on Validation and Test Sets

In [22]:
# Drive folder
DRIVE_MODEL_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/bart_headline_generation_model/final_model"

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# Validation evaluation
val_metrics = trainer.evaluate(
    eval_dataset=tokenized_val,
    metric_key_prefix='val'
)

print('Validation metrics:')
print(json.dumps(val_metrics, indent=2))

# Test evaluation
test_metrics = trainer.evaluate(
    eval_dataset=tokenized_test,
    metric_key_prefix='test'
)

print('Test metrics:')
print(json.dumps(test_metrics, indent=2))

# Save metrics directly into Drive
metrics_path = os.path.join(DRIVE_MODEL_DIR, 'final_metrics.json')

with open(metrics_path, 'w') as f:
    json.dump(
        {
            'validation': val_metrics,
            'test': test_metrics
        },
        f,
        indent=2
    )

print('Saved metrics to:')
print(metrics_path)

[transformers] early stopping required metric_for_best_model, but did not find eval_rougeL so early stopping is disabled


Training Loss,Validation Loss,Step,Rouge1,Rouge2,Rougel
14.413197,1.839970,4760,0.446843,0.223274,0.407352


Validation metrics:
{
  "val_loss": 1.8399697542190552,
  "val_rouge1": 0.44684252439837935,
  "val_rouge2": 0.22327400002452819,
  "val_rougeL": 0.4073518209269591
}


[transformers] early stopping required metric_for_best_model, but did not find eval_rougeL so early stopping is disabled


Training Loss,Validation Loss,Step,Rouge1,Rouge2,Rougel
14.413197,1.830061,4760,0.450392,0.223219,0.411631


Test metrics:
{
  "test_loss": 1.8300608396530151,
  "test_rouge1": 0.4503924796413583,
  "test_rouge2": 0.2232189597960188,
  "test_rougeL": 0.41163069586263973
}
Saved metrics to:
/content/drive/MyDrive/headlineGenerationProjectNLP/bart_headline_generation_model/final_model/final_metrics.json


## 15. Generate Sample Headlines

In [23]:
# Drive folder
DRIVE_MODEL_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/bart_headline_generation_model/final_model"

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

def generate_headline(
    article_text,
    max_input_length=1024,
    max_output_length=64
):
    model.eval()

    inputs = tokenizer(
        str(article_text),
        return_tensors='pt',
        truncation=True,
        max_length=max_input_length,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_output_length,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    headline = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    return headline


# Generate predictions
sample_df = test_df.sample(
    min(5, len(test_df)),
    random_state=42
).copy()

generated_results = []

for i, row in sample_df.iterrows():

    pred = generate_headline(row['model_input'])

    print('=' * 100)

    print('ARTICLE PREVIEW:\n')
    print(row['model_input'][:700])

    print('\nREAL HEADLINE:\n')
    print(row['model_target'])

    print('\nPREDICTED HEADLINE:\n')
    print(pred)

    generated_results.append({
        'article_preview': row['model_input'][:700],
        'real_headline': row['model_target'],
        'predicted_headline': pred
    })


# Save generated examples directly into Drive
results_df = pd.DataFrame(generated_results)

results_path = os.path.join(
    DRIVE_MODEL_DIR,
    'generated_headlines_examples.csv'
)

results_df.to_csv(results_path, index=False)

print('\nSaved generated headline examples to:')
print(results_path)

ARTICLE PREVIEW:

summarize: Recently, The Verge reported on a string of ransomware attacks that have hit cities including Baltimore; Atlanta, Georgia; Newark, New Jersey; and 22 Texas towns. Even The Weather Channel has fallen victim. But before those attacks, there was an attack on the nation's capital, days before the presidential inauguration. An article from The Wall Street Journal details how hackers Alexandru Isvanca and Eveline Cismaru seized control of Washington, DC's surveillance cameras right before Trump's inauguration. The piece is full of twists and turns, from the small-time beginnings of the hackers' scamming careers to them eventually turning on each other. The story contains a lot of colorf

REAL HEADLINE:

Hackers seized cameras before Trump's inauguration and left a smoking gun behind

PREDICTED HEADLINE:

How hackers seized control of DC surveillance cameras before Trump's inauguration
ARTICLE PREVIEW:

summarize: Many of us are feeling helpless as we watch storie

In [24]:
def predict_manual_article():
    print("Paste your article below.")
    print("When you finish, press ENTER twice:\n")

    lines = []
    while True:
        line = input()
        if line == "":
            break
        lines.append(line)

    article_text = "\n".join(lines).strip()

    if len(article_text) == 0:
        print("No article entered.")
        return

    predicted_headline = generate_headline(article_text)

    print("\n" + "=" * 80)
    print("MANUAL ARTICLE PREVIEW:\n")
    print(article_text[:1000])

    print("\nPREDICTED HEADLINE:\n")
    print(predicted_headline)
    print("=" * 80)

    return predicted_headline


manual_prediction = predict_manual_article()

Paste your article below.
When you finish, press ENTER twice:

summarize: (Repeats to additional subscribers without change to text) * Cyprus-based insurer put into liquidation * Public anger as 200,000 Bulgarians left without car insurance * Government calls for resignation of regulator's deputy chair By Angel Krasimirov SOFIA, Aug 20 (Reuters) - Bulgaria's chief prosecutor ordered the country's main intelligence agency on Monday to investigate financial regulator KFN's oversight of the Bulgarian branch of Cyprus-based Olympic Insurance Company Ltd, after the firm was placed in liquidation. The Financial Supervision Commission, or KFN, said it prohibited Olympic Insurance's branch in Bulgaria from signing new contracts on May 10 after it was informed about problems at the insurer by Cyprus's Insurance Companies Control Service. The Cypriot insurance regulator appointed a provisional liquidator for Olympic Insurance, which operated in Cyprus and Bulgaria, on Aug. 10, after revoking the